# Лабораторная работа 2 — Методы выделения и улучшения границ (вариант 15)

Исходное изображение: `lab2/image_v1-15.png`.

Цель: применить различные операторы для выделения границ и улучшения изображений.


In [ ]:
import os
import numpy as np
import cv2
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt

IMAGE_PATH = "/Users/23108022/Documents/repositories/mephi-computer-vision-2025/labs/lab2/lab2/image_v1-15.png"
assert os.path.exists(IMAGE_PATH), "Image not found"

# read as BGR then convert to RGB for correct display
bgr = cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

print(f"Размерность изображения: {rgb.shape}")
print(f"Число цветовых каналов: {rgb.shape[2] if len(rgb.shape) == 3 else 1}")
print(f"Яркостное разрешение: 1 байт (диапазон [0, 255])")
print(f"dtype={rgb.dtype}, min={rgb.min()}, max={rgb.max()}, mean={rgb.mean():.2f}")
display(Image.fromarray(rgb))


## 1. Загрузка и визуализация изображения

### а) Числовые характеристики изображения
Приведено выше.


In [ ]:
# б) Приведение к одноканальному изображению в градациях серого
print(f"Исходное изображение: RGB, shape={rgb.shape}")
print(f"Изображение в градациях серого: shape={gray.shape}, min={gray.min()}, max={gray.max()}, mean={gray.mean():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(rgb)
axes[0].set_title('Исходное изображение (RGB)')
axes[0].axis('off')
axes[1].imshow(gray, cmap='gray')
axes[1].set_title('Изображение в градациях серого')
axes[1].axis('off')
plt.tight_layout()
plt.show()


## 2. Градиентные операторы

Вспомогательная функция для нормализации изображений к диапазону [0, 255] и применения выравнивания гистограмм для визуализации.


In [ ]:
def to_uint8_image(image):
    """Нормализация изображения к диапазону [0, 255] и применение выравнивания гистограмм."""
    # Нормализация к [0, 255]
    res = (image - image.min()) * 255.0 / float(max(1, image.ptp()))
    res = res.astype(np.uint8)
    # Выравнивание гистограммы для лучшей визуализации
    res = cv2.equalizeHist(res)
    return res

def compute_gradient_angle(gx, gy):
    """Вычисление угла наклона градиента в градусах."""
    angle = np.arctan2(gy, gx)
    return np.degrees(angle)


### а) Оператор Робертса


In [ ]:
# Оператор Робертса
kernel_roberts_x = np.array([[1,  0],
                             [0, -1]], dtype=np.float32)
kernel_roberts_y = np.array([[0,  1],
                             [-1, 0]], dtype=np.float32)

# Применение оператора Робертса
roberts_gx = cv2.filter2D(gray.astype(np.float32), cv2.CV_32F, kernel_roberts_x)
roberts_gy = cv2.filter2D(gray.astype(np.float32), cv2.CV_32F, kernel_roberts_y)
roberts_magnitude = np.sqrt(roberts_gx**2 + roberts_gy**2)
roberts_angle = compute_gradient_angle(roberts_gx, roberts_gy)

# Нормализация для визуализации
roberts_gx_viz = to_uint8_image(roberts_gx)
roberts_gy_viz = to_uint8_image(roberts_gy)
roberts_mag_viz = to_uint8_image(roberts_magnitude)
roberts_angle_viz = to_uint8_image(roberts_angle + 180)  # Сдвиг для нормализации

# Визуализация
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Исходное изображение')
axes[0, 0].axis('off')

axes[0, 1].imshow(roberts_gx_viz, cmap='gray')
axes[0, 1].set_title('Горизонтальный градиент (Робертс)')
axes[0, 1].axis('off')

axes[0, 2].imshow(roberts_gy_viz, cmap='gray')
axes[0, 2].set_title('Вертикальный градиент (Робертс)')
axes[0, 2].axis('off')

axes[1, 0].imshow(roberts_mag_viz, cmap='gray')
axes[1, 0].set_title('Модуль градиента (Робертс)')
axes[1, 0].axis('off')

axes[1, 1].imshow(roberts_angle_viz, cmap='gray')
axes[1, 1].set_title('Угол наклона градиента (Робертс)')
axes[1, 1].axis('off')

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


### б) Оператор Превитта


In [ ]:
# Оператор Превитта
kernel_prewitt_x = np.array([[-1, 0, 1],
                             [-1, 0, 1],
                             [-1, 0, 1]], dtype=np.float32)
kernel_prewitt_y = np.array([[1,  1,  1],
                             [0,  0,  0],
                             [-1, -1, -1]], dtype=np.float32)

# Применение оператора Превитта
prewitt_gx = cv2.filter2D(gray.astype(np.float32), cv2.CV_32F, kernel_prewitt_x)
prewitt_gy = cv2.filter2D(gray.astype(np.float32), cv2.CV_32F, kernel_prewitt_y)
prewitt_magnitude = np.sqrt(prewitt_gx**2 + prewitt_gy**2)
prewitt_angle = compute_gradient_angle(prewitt_gx, prewitt_gy)

# Нормализация для визуализации
prewitt_gx_viz = to_uint8_image(prewitt_gx)
prewitt_gy_viz = to_uint8_image(prewitt_gy)
prewitt_mag_viz = to_uint8_image(prewitt_magnitude)
prewitt_angle_viz = to_uint8_image(prewitt_angle + 180)

# Визуализация
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Исходное изображение')
axes[0, 0].axis('off')

axes[0, 1].imshow(prewitt_gx_viz, cmap='gray')
axes[0, 1].set_title('Горизонтальный градиент (Превитт)')
axes[0, 1].axis('off')

axes[0, 2].imshow(prewitt_gy_viz, cmap='gray')
axes[0, 2].set_title('Вертикальный градиент (Превитт)')
axes[0, 2].axis('off')

axes[1, 0].imshow(prewitt_mag_viz, cmap='gray')
axes[1, 0].set_title('Модуль градиента (Превитт)')
axes[1, 0].axis('off')

axes[1, 1].imshow(prewitt_angle_viz, cmap='gray')
axes[1, 1].set_title('Угол наклона градиента (Превитт)')
axes[1, 1].axis('off')

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


### в) Оператор Собеля


In [ ]:
# Оператор Собеля (используем встроенную функцию OpenCV)
sobel_gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
sobel_gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
sobel_magnitude = np.sqrt(sobel_gx**2 + sobel_gy**2)
sobel_angle = compute_gradient_angle(sobel_gx, sobel_gy)

# Нормализация для визуализации
sobel_gx_viz = to_uint8_image(sobel_gx)
sobel_gy_viz = to_uint8_image(sobel_gy)
sobel_mag_viz = to_uint8_image(sobel_magnitude)
sobel_angle_viz = to_uint8_image(sobel_angle + 180)

# Визуализация
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Исходное изображение')
axes[0, 0].axis('off')

axes[0, 1].imshow(sobel_gx_viz, cmap='gray')
axes[0, 1].set_title('Горизонтальный градиент (Собель)')
axes[0, 1].axis('off')

axes[0, 2].imshow(sobel_gy_viz, cmap='gray')
axes[0, 2].set_title('Вертикальный градиент (Собель)')
axes[0, 2].axis('off')

axes[1, 0].imshow(sobel_mag_viz, cmap='gray')
axes[1, 0].set_title('Модуль градиента (Собель)')
axes[1, 0].axis('off')

axes[1, 1].imshow(sobel_angle_viz, cmap='gray')
axes[1, 1].set_title('Угол наклона градиента (Собель)')
axes[1, 1].axis('off')

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("Выводы:")
print("- Оператор Робертса: самый простой (2x2), чувствителен к шуму, быстрый.")
print("- Оператор Превитта: 3x3, лучше подавляет шум, чем Робертс.")
print("- Оператор Собеля: 3x3, взвешенный градиент, лучше всего подходит для выделения границ.")


## 3. Фильтр Гаусса с различными значениями среднеквадратического отклонения


In [ ]:
# Применение фильтра Гаусса с различными значениями σ
sigmas = [0.5, 5, 10, 20]
gaussian_results = {}

for sigma in sigmas:
    # Размер ядра выбираем как 6*sigma + 1 для обеспечения достаточного покрытия
    ksize = int(6 * sigma + 1)
    if ksize % 2 == 0:
        ksize += 1
    blurred = cv2.GaussianBlur(gray, (ksize, ksize), sigma)
    gaussian_results[sigma] = blurred

# Визуализация
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Исходное изображение')
axes[0, 0].axis('off')

for idx, sigma in enumerate(sigmas, 1):
    row = idx // 3
    col = idx % 3
    axes[row, col].imshow(gaussian_results[sigma], cmap='gray')
    axes[row, col].set_title(f'σ = {sigma}')
    axes[row, col].axis('off')

# Скрыть последнюю пустую ячейку
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("Выводы:")
print("- σ = 0.5: минимальное сглаживание, почти незаметно.")
print("- σ = 5: умеренное сглаживание, сохраняются основные детали.")
print("- σ = 10: сильное сглаживание, мелкие детали теряются.")
print("- σ = 20: очень сильное сглаживание, изображение сильно размыто.")


## 4. Оператор Собеля к изображению, сглаженному фильтром Гаусса


In [ ]:
# Применение оператора Собеля к изображению, сглаженному фильтром Гаусса
sigmas_sobel = [0.5, 2, 5, 10]
sobel_gaussian_results = {}

for sigma in sigmas_sobel:
    # Сглаживание Гауссом
    ksize = int(6 * sigma + 1)
    if ksize % 2 == 0:
        ksize += 1
    blurred = cv2.GaussianBlur(gray, (ksize, ksize), sigma)
    
    # Применение оператора Собеля
    sobel_gx = cv2.Sobel(blurred, cv2.CV_32F, 1, 0, ksize=3)
    sobel_gy = cv2.Sobel(blurred, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobel_gx**2 + sobel_gy**2)
    angle = compute_gradient_angle(sobel_gx, sobel_gy)
    
    sobel_gaussian_results[sigma] = {
        'magnitude': magnitude,
        'angle': angle
    }

# Визуализация модулей градиентов
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for idx, sigma in enumerate(sigmas_sobel):
    row = idx // 2
    col = idx % 2
    mag_viz = to_uint8_image(sobel_gaussian_results[sigma]['magnitude'])
    axes[row, col].imshow(mag_viz, cmap='gray')
    axes[row, col].set_title(f'Модуль градиента, σ = {sigma}')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

# Визуализация углов градиентов
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for idx, sigma in enumerate(sigmas_sobel):
    row = idx // 2
    col = idx % 2
    angle_viz = to_uint8_image(sobel_gaussian_results[sigma]['angle'] + 180)
    axes[row, col].imshow(angle_viz, cmap='gray')
    axes[row, col].set_title(f'Угол наклона градиента, σ = {sigma}')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("Выводы:")
print("- Предварительное сглаживание Гауссом уменьшает шум, но может размыть границы.")
print("- Меньшие σ сохраняют больше деталей, но могут выделять шум.")
print("- Большие σ лучше подавляют шум, но могут потерять тонкие границы.")


## 5. Фильтр лапласиана гауссиана (LoG)


In [ ]:
def create_log_kernel(k, sigma):
    """Создание ядра фильтра LoG (Laplacian of Gaussian)."""
    kernel = np.zeros((k, k), dtype=np.float32)
    center = k // 2
    
    for i in range(k):
        for j in range(k):
            x = i - center
            y = j - center
            r_sq = x**2 + y**2
            kernel[i, j] = -(1 / (np.pi * sigma**4)) * (1 - r_sq / (2 * sigma**2)) * np.exp(-r_sq / (2 * sigma**2))
    
    return kernel

def apply_log_filter(image, k, sigma):
    """Применение фильтра LoG к изображению."""
    log_kernel = create_log_kernel(k, sigma)
    log_result = cv2.filter2D(image.astype(np.float32), cv2.CV_32F, log_kernel)
    return log_result, log_kernel

def zero_crossing(log_image):
    """Вычисление пересечений нуля для LoG."""
    zc_image = np.zeros(log_image.shape, dtype=np.uint8)
    H, W = log_image.shape
    
    for i in range(1, H - 1):
        for j in range(1, W - 1):
            neighbors = [
                log_image[i-1, j-1], log_image[i-1, j], log_image[i-1, j+1],
                log_image[i, j-1], log_image[i, j+1],
                log_image[i+1, j-1], log_image[i+1, j], log_image[i+1, j+1]
            ]
            
            # Проверка пересечения нуля
            if log_image[i, j] == 0:
                # Проверка наличия разных знаков в соседях
                positive = sum(1 for n in neighbors if n > 0)
                negative = sum(1 for n in neighbors if n < 0)
                if positive > 0 and negative > 0:
                    zc_image[i, j] = 255
            elif log_image[i, j] < 0:
                # Текущий пиксель отрицательный - ищем положительных соседей
                if any(n > 0 for n in neighbors):
                    zc_image[i, j] = 255
            else:
                # Текущий пиксель положительный - ищем отрицательных соседей
                if any(n < 0 for n in neighbors):
                    zc_image[i, j] = 255
    
    return zc_image


### а) k = 5, σ = 0.2


In [ ]:
# k = 5, σ = 0.2
k1, sigma1 = 5, 0.2
log_result_1, log_kernel_1 = apply_log_filter(gray, k1, sigma1)
log_binary_1 = (log_result_1 > 0).astype(np.uint8) * 255
zc_image_1 = zero_crossing(log_result_1)

# Визуализация
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Исходное изображение')
axes[0, 0].axis('off')

axes[0, 1].imshow(log_kernel_1, cmap='gray')
axes[0, 1].set_title(f'Ядро k = {k1}, σ = {sigma1}')
axes[0, 1].axis('off')

axes[0, 2].imshow(to_uint8_image(log_result_1), cmap='gray')
axes[0, 2].set_title('Результат применения фильтра LoG')
axes[0, 2].axis('off')

axes[1, 0].imshow(log_binary_1, cmap='gray')
axes[1, 0].set_title('Бинаризованный результат')
axes[1, 0].axis('off')

axes[1, 1].imshow(zc_image_1, cmap='gray')
axes[1, 1].set_title('Бинарное изображение пересечений нуля')
axes[1, 1].axis('off')

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


### б) k = 10, σ = 2


In [ ]:
# k = 10, σ = 2
k2, sigma2 = 10, 2.0
log_result_2, log_kernel_2 = apply_log_filter(gray, k2, sigma2)
log_binary_2 = (log_result_2 > 0).astype(np.uint8) * 255
zc_image_2 = zero_crossing(log_result_2)

# Визуализация
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Исходное изображение')
axes[0, 0].axis('off')

axes[0, 1].imshow(log_kernel_2, cmap='gray')
axes[0, 1].set_title(f'Ядро k = {k2}, σ = {sigma2}')
axes[0, 1].axis('off')

axes[0, 2].imshow(to_uint8_image(log_result_2), cmap='gray')
axes[0, 2].set_title('Результат применения фильтра LoG')
axes[0, 2].axis('off')

axes[1, 0].imshow(log_binary_2, cmap='gray')
axes[1, 0].set_title('Бинаризованный результат')
axes[1, 0].axis('off')

axes[1, 1].imshow(zc_image_2, cmap='gray')
axes[1, 1].set_title('Бинарное изображение пересечений нуля')
axes[1, 1].axis('off')

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


### в) k = 30, σ = 2


In [ ]:
# k = 30, σ = 2
k3, sigma3 = 30, 2.0
log_result_3, log_kernel_3 = apply_log_filter(gray, k3, sigma3)
log_binary_3 = (log_result_3 > 0).astype(np.uint8) * 255
zc_image_3 = zero_crossing(log_result_3)

# Визуализация
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Исходное изображение')
axes[0, 0].axis('off')

axes[0, 1].imshow(log_kernel_3, cmap='gray')
axes[0, 1].set_title(f'Ядро k = {k3}, σ = {sigma3}')
axes[0, 1].axis('off')

axes[0, 2].imshow(to_uint8_image(log_result_3), cmap='gray')
axes[0, 2].set_title('Результат применения фильтра LoG')
axes[0, 2].axis('off')

axes[1, 0].imshow(log_binary_3, cmap='gray')
axes[1, 0].set_title('Бинаризованный результат')
axes[1, 0].axis('off')

axes[1, 1].imshow(zc_image_3, cmap='gray')
axes[1, 1].set_title('Бинарное изображение пересечений нуля')
axes[1, 1].axis('off')

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


### г) k = 30, σ = 10


In [ ]:
# k = 30, σ = 10
k4, sigma4 = 30, 10.0
log_result_4, log_kernel_4 = apply_log_filter(gray, k4, sigma4)
log_binary_4 = (log_result_4 > 0).astype(np.uint8) * 255
zc_image_4 = zero_crossing(log_result_4)

# Визуализация
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Исходное изображение')
axes[0, 0].axis('off')

axes[0, 1].imshow(log_kernel_4, cmap='gray')
axes[0, 1].set_title(f'Ядро k = {k4}, σ = {sigma4}')
axes[0, 1].axis('off')

axes[0, 2].imshow(to_uint8_image(log_result_4), cmap='gray')
axes[0, 2].set_title('Результат применения фильтра LoG')
axes[0, 2].axis('off')

axes[1, 0].imshow(log_binary_4, cmap='gray')
axes[1, 0].set_title('Бинаризованный результат')
axes[1, 0].axis('off')

axes[1, 1].imshow(zc_image_4, cmap='gray')
axes[1, 1].set_title('Бинарное изображение пересечений нуля')
axes[1, 1].axis('off')

axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("Выводы:")
print("- Увеличение k при фиксированном σ улучшает покрытие, но увеличивает вычислительную сложность.")
print("- Увеличение σ увеличивает область влияния фильтра, лучше подавляет шум, но может размыть границы.")
print("- Метод пересечений нуля позволяет найти границы без использования порогов.")


## 6. Фильтр разности гауссиан (DoG)


In [ ]:
def apply_dog_filter(image, sigma1, sigma2):
    """Применение фильтра DoG (Difference of Gaussians)."""
    # Размер ядра выбирается автоматически на основе большего sigma
    sigma_max = max(sigma1, sigma2)
    ksize = int(6 * sigma_max + 1)
    if ksize % 2 == 0:
        ksize += 1
    
    gauss1 = cv2.GaussianBlur(image, (ksize, ksize), sigma1)
    gauss2 = cv2.GaussianBlur(image, (ksize, ksize), sigma2)
    
    dog_result = gauss1.astype(np.float32) - gauss2.astype(np.float32)
    return dog_result

# Применение DoG с различными параметрами
dog_params = [
    (2, 1.6),
    (5, 1.6),
    (2, 5),
    (5, 3)  # Экспериментально подобранные значения
]

dog_results = {}
for sigma1, sigma2 in dog_params:
    dog_result = apply_dog_filter(gray, sigma1, sigma2)
    dog_results[(sigma1, sigma2)] = dog_result

# Визуализация
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for idx, (sigma1, sigma2) in enumerate(dog_params):
    row = idx // 2
    col = idx % 2
    dog_viz = to_uint8_image(dog_results[(sigma1, sigma2)])
    axes[row, col].imshow(dog_viz, cmap='gray')
    axes[row, col].set_title(f'Ядро σ₁ = {sigma1}, σ₂ = {sigma2}')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("Выводы:")
print("- DoG аппроксимирует LoG и является более вычислительно эффективным.")
print("- Разница σ₁ и σ₂ определяет ширину полосы пропускания фильтра.")
print("- Меньшая разница выделяет более тонкие детали, большая разница - более грубые структуры.")


## 7. Выделение границ на изображении

### а) Детектор Марра-Хилдрета


In [ ]:
def marr_hildreth_detector(image, sigma1, sigma2):
    """Детектор Марра-Хилдрета на основе DoG."""
    # Вычисляем DoG
    dog_result = apply_dog_filter(image, sigma1, sigma2)
    
    # Поиск пересечений нуля
    zc_image = zero_crossing(dog_result)
    
    return dog_result, zc_image

# Применение детектора Марра-Хилдрета с различными параметрами
mh_params = [
    (2, 1.6),
    (5, 1.6),
    (2, 5),
    (5, 3)
]

mh_results = {}
for sigma1, sigma2 in mh_params:
    dog_result, zc_image = marr_hildreth_detector(gray, sigma1, sigma2)
    mh_results[(sigma1, sigma2)] = {
        'dog': dog_result,
        'zero_crossing': zc_image
    }

# Визуализация
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, (sigma1, sigma2) in enumerate(mh_params):
    dog_viz = to_uint8_image(mh_results[(sigma1, sigma2)]['dog'])
    zc_image = mh_results[(sigma1, sigma2)]['zero_crossing']
    
    axes[0, idx].imshow(dog_viz, cmap='gray')
    axes[0, idx].set_title(f'DoG: σ₁ = {sigma1}, σ₂ = {sigma2}')
    axes[0, idx].axis('off')
    
    axes[1, idx].imshow(zc_image, cmap='gray')
    axes[1, idx].set_title(f'Границы (пересечения нуля)')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.show()


### б) Детектор Кэнни


In [ ]:
# Применение детектора Кэнни с различными параметрами
# Пороги подобраны экспериментально для данного изображения
canny_params = [
    (50, 150, 0.5),
    (50, 150, 2),
    (50, 150, 5),
    (80, 200, 10)
]

canny_results = {}
for t1, t2, sigma in canny_params:
    # Сглаживание Гауссом перед применением Кэнни
    ksize = int(6 * sigma + 1)
    if ksize % 2 == 0:
        ksize += 1
    blurred = cv2.GaussianBlur(gray, (ksize, ksize), sigma)
    
    # Применение детектора Кэнни
    edges = cv2.Canny(blurred, t1, t2)
    canny_results[(t1, t2, sigma)] = edges

# Визуализация
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for idx, (t1, t2, sigma) in enumerate(canny_params):
    row = idx // 2
    col = idx % 2
    axes[row, col].imshow(canny_results[(t1, t2, sigma)], cmap='gray')
    axes[row, col].set_title(f't1 = {t1}, t2 = {t2}, σ = {sigma}')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("Выводы:")
print("- Детектор Марра-Хилдрета: использует пересечения нуля LoG/DoG, не требует порогов.")
print("- Детектор Кэнни: многоэтапный алгоритм с подавлением немаксимумов и гистерезисом.")
print("- Кэнни дает более тонкие и непрерывные границы, но требует подбора порогов.")
print("- Марра-Хилдрета более устойчив к изменению освещения.")


## 8. Нерезкое маскирование изображения

Формула нерезкого маскирования: $g = a + \lambda d = a + \lambda(a - \tilde{a})$, где:
- $a$ - исходное изображение
- $\tilde{a}$ - сглаженное изображение (Гауссом)
- $\lambda$ - коэффициент усиления
- $t$ - порог (опционально)


### а) Без использования порога (t = 0)


In [ ]:
def unsharp_masking(image, sigma, lambda_val, threshold=0):
    """Нерезкое маскирование изображения."""
    # Сглаживание Гауссом
    ksize = int(6 * sigma + 1)
    if ksize % 2 == 0:
        ksize += 1
    blurred = cv2.GaussianBlur(image.astype(np.float32), (ksize, ksize), sigma)
    
    # Вычисление деталей (разность оригинал - сглаженный)
    details = image.astype(np.float32) - blurred
    
    # Применение порога, если указан
    if threshold > 0:
        details = np.where(np.abs(details) > threshold, details, 0)
    
    # Усиление деталей
    sharpened = image.astype(np.float32) + lambda_val * details
    
    # Обрезка к допустимому диапазону [0, 255]
    sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)
    
    return sharpened, details

# Параметры для нерезкого маскирования без порога
um_params_no_threshold = [
    (2, 2),
    (20, 2),
    (2, 10),
    (10, 5)
]

um_results_no_thresh = {}
for sigma, lambda_val in um_params_no_threshold:
    sharpened, details = unsharp_masking(gray, sigma, lambda_val, threshold=0)
    um_results_no_thresh[(sigma, lambda_val)] = {
        'sharpened': sharpened,
        'details': details
    }

# Визуализация
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, (sigma, lambda_val) in enumerate(um_params_no_threshold):
    sharp = um_results_no_thresh[(sigma, lambda_val)]['sharpened']
    details_viz = to_uint8_image(np.abs(um_results_no_thresh[(sigma, lambda_val)]['details']))
    
    axes[0, idx].imshow(sharp, cmap='gray')
    axes[0, idx].set_title(f'Усиленное: σ = {sigma}, λ = {lambda_val}')
    axes[0, idx].axis('off')
    
    axes[1, idx].imshow(details_viz, cmap='gray')
    axes[1, idx].set_title(f'Детали (маска)')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.show()


### б) С использованием порога (t > 0)


In [ ]:
# Параметры для нерезкого маскирования с порогом
# Пороги подобраны экспериментально
um_params_with_threshold = [
    (2, 2, 5),
    (20, 2, 10),
    (2, 10, 8),
    (10, 5, 12)
]

um_results_with_thresh = {}
for sigma, lambda_val, threshold in um_params_with_threshold:
    sharpened, details = unsharp_masking(gray, sigma, lambda_val, threshold=threshold)
    um_results_with_thresh[(sigma, lambda_val, threshold)] = {
        'sharpened': sharpened,
        'details': details
    }

# Визуализация
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, (sigma, lambda_val, threshold) in enumerate(um_params_with_threshold):
    sharp = um_results_with_thresh[(sigma, lambda_val, threshold)]['sharpened']
    details_viz = to_uint8_image(np.abs(um_results_with_thresh[(sigma, lambda_val, threshold)]['details']))
    
    axes[0, idx].imshow(sharp, cmap='gray')
    axes[0, idx].set_title(f'Усиленное: σ = {sigma}, λ = {lambda_val}, t = {threshold}')
    axes[0, idx].axis('off')
    
    axes[1, idx].imshow(details_viz, cmap='gray')
    axes[1, idx].set_title(f'Детали (маска с порогом)')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.show()

print("Выводы:")
print("- Нерезкое маскирование усиливает высокочастотные компоненты изображения.")
print("- Увеличение λ усиливает эффект, но может привести к артефактам.")
print("- Большие σ сглаживают больше, выделяя крупные детали.")
print("- Порог t позволяет подавить слабые детали и уменьшить шум.")
print("- Комбинация параметров должна подбираться экспериментально для каждого изображения.")
